# Self-Contained Jupyter Notebook: Chapter 2 — The Agent Engineer's Toolkit

This notebook converts the **most important implementation ideas** from Chapter 2 into a runnable, self-sufficient notebook.

## What this notebook covers
The chapter emphasizes three highly practical engineering ideas:

1. **Tool-using agents**  
   An agent can choose between tools such as a calculator or a web-search-like function.

2. **Retrieval as memory infrastructure**  
   Documents are chunked, indexed, and retrieved to support grounded answers.

3. **Routing / orchestration**  
   Different query types can be routed to different handlers or model strategies.

To keep the notebook self-contained, this version uses:
- pure Python
- `scikit-learn` TF-IDF vectors for retrieval
- simple tool abstractions
- a lightweight query router
- explicit provenance

No API key is required.

## Why these snippets?
Chapter 2 presents:
- a **LangChain tool-using agent** example with calculator and web search
- discussion of **frameworks** such as LangChain and LangGraph
- a strong section on **vector databases and RAG**
- a **hybrid model routing** example for different query types

This notebook turns those core ideas into one coherent learning artifact.

In [ ]:
# Optional, if needed:
# !pip install scikit-learn pandas numpy

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Callable, Any, Tuple
import math
import re
import textwrap

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1) Create a small local knowledge base

The chapter discusses agent frameworks, vector search, chunking, tool abstractions, and cloud-native platforms.
We create a compact corpus that lets us demonstrate retrieval and grounding.

In [ ]:
knowledge_base = {
    "frameworks.txt": '''
LangChain is a modular framework for building LLM pipelines, tools, retrievers, and memory.
It is strong for orchestration and tool workflows, but more complex multi-agent systems often need LangGraph.

LangGraph extends LangChain with stateful, cyclical workflows.
It represents workflows as nodes, edges, and state, and supports loops, conditional routing, and long-running processes.

LlamaIndex is especially strong for document retrieval, indexing, and knowledge-centric applications.
It works well as the memory cortex inside a larger agent system.
''',

    "retrieval.txt": '''
Vector databases transform chunks of text into vectors and support semantic retrieval.
A standard RAG flow is:
1. chunk documents
2. embed or vectorize chunks
3. store them with metadata
4. retrieve relevant chunks on demand
5. inject them into the prompt or answer construction

Chunking is critical.
If chunks are too small, important context may be lost.
If chunks are too large, the signal may be diluted by noise.
Reranking and metadata filtering often improve retrieval precision.
''',

    "tools.txt": '''
Tool integration lets an agent act outside the model itself.
A tool abstraction wraps a function with a name, description, and callable interface.

Examples include:
- calculator tools
- stock or weather lookup tools
- search tools
- database query tools
- code execution tools

OpenAI function calling uses JSON schemas to describe tools.
LangChain tools wrap Python functions so that an agent can discover and invoke them.
''',

    "models.txt": '''
Model selection shapes what an agent can understand and generate.
Some systems use hybrid routing, where different query types are handled by different models or strategies.

For example:
- factual queries may go to a fast and cheap model
- creative queries may go to a model optimized for style
- analytical queries may go to a stronger reasoning model

This routing layer optimizes cost, latency, and capability.
''',

    "cloud.txt": '''
Major cloud providers offer managed agent platforms.
AWS emphasizes breadth of services and Bedrock-based agent workflows.
Azure emphasizes enterprise governance and deep Microsoft integration.
Google Cloud emphasizes interoperability, enterprise search, and strong AI infrastructure.

Platform choice depends on existing ecosystem alignment, governance needs, latency constraints, and deployment style.
'''
}

## 2) Build chunks for retrieval

In [ ]:
@dataclass
class Chunk:
    chunk_id: str
    source: str
    text: str
    start_char: int
    end_char: int

def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def chunk_text(text: str, chunk_size: int = 320, chunk_overlap: int = 60) -> List[Tuple[str, int, int]]:
    text = normalize_whitespace(text)
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + chunk_size, n)
        piece = text[start:end]

        if end < n:
            sentence_break = max(piece.rfind(". "), piece.rfind("? "), piece.rfind("! "))
            word_break = piece.rfind(" ")
            best_break = max(sentence_break, word_break)
            if best_break > chunk_size // 2:
                end = start + best_break + 1
                piece = text[start:end]

        chunks.append((piece.strip(), start, end))

        if end >= n:
            break

        start = max(0, end - chunk_overlap)

    return chunks

def build_chunks(corpus: Dict[str, str], chunk_size: int = 320, chunk_overlap: int = 60) -> List[Chunk]:
    items = []
    i = 1
    for source, text in corpus.items():
        for piece, start, end in chunk_text(text, chunk_size, chunk_overlap):
            items.append(Chunk(
                chunk_id=f"C{i:03d}",
                source=source,
                text=piece,
                start_char=start,
                end_char=end
            ))
            i += 1
    return items

chunks = build_chunks(knowledge_base)
len(chunks)

In [ ]:
for c in chunks[:6]:
    print(f"{c.chunk_id} | {c.source} | {c.start_char}-{c.end_char}")
    print(textwrap.shorten(c.text, width=120, placeholder=" ..."))
    print("-" * 80)

## 3) Build a simple vector index

In [ ]:
class SimpleVectorIndex:
    def __init__(self, chunks: List[Chunk]):
        self.chunks = chunks
        self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
        self.matrix = self.vectorizer.fit_transform([c.text for c in chunks])

    def search(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        q = self.vectorizer.transform([query])
        sims = cosine_similarity(q, self.matrix).flatten()
        idxs = np.argsort(sims)[::-1][:top_k]

        results = []
        for idx in idxs:
            results.append({
                "score": float(sims[idx]),
                "chunk": self.chunks[idx]
            })
        return results

index = SimpleVectorIndex(chunks)

## 4) Add a few tools

In [ ]:
@dataclass
class Tool:
    name: str
    description: str
    func: Callable[[str], str]

def calculator_tool(expression: str) -> str:
    cleaned = re.sub(r"[^0-9\+\-\*\/\(\)\.\s]", "", expression)
    if not cleaned.strip():
        return "No valid mathematical expression found."
    try:
        result = eval(cleaned, {"__builtins__": {}}, {})
        return f"Result: {result}"
    except Exception as e:
        return f"Calculation error: {e}"

mock_web = {
    "langchain": "LangChain is widely used for LLM orchestration, tools, retrievers, and memory.",
    "langgraph": "LangGraph is used for stateful workflows with nodes, edges, loops, and conditional routing.",
    "vector database": "Vector databases support semantic retrieval using embeddings or vectorized representations.",
    "azure": "Azure AI platforms emphasize enterprise governance and Microsoft ecosystem integration.",
    "aws": "AWS Bedrock supports managed foundation model access and agent workflows."
}

def web_search_tool(query: str) -> str:
    q = query.lower()
    matches = []
    for k, v in mock_web.items():
        if k in q or any(token in v.lower() for token in q.split()):
            matches.append(f"{k}: {v}")
    if matches:
        return "\n".join(matches[:3])
    return "No useful mock web result found."

tools = {
    "Calculator": Tool(
        name="Calculator",
        description="Useful for arithmetic and simple mathematical calculations.",
        func=calculator_tool
    ),
    "WebSearch": Tool(
        name="WebSearch",
        description="Useful for mock current-information lookups and quick external facts.",
        func=web_search_tool
    )
}

## 5) Query classification / routing

In [ ]:
def classify_query(query: str) -> str:
    q = query.lower()

    math_patterns = [
        r"\d+\s*[\+\-\*\/]\s*\d+",
        r"square root",
        r"calculate",
        r"what is \d"
    ]
    if any(re.search(p, q) for p in math_patterns):
        return "math"

    retrieval_keywords = [
        "framework", "langchain", "langgraph", "llamaindex",
        "vector", "retrieval", "rag", "chunk", "tool", "function calling",
        "azure", "aws", "google cloud", "model selection"
    ]
    if any(k in q for k in retrieval_keywords):
        return "knowledge"

    if any(word in q for word in ["write", "draft", "creative", "story"]):
        return "creative"

    return "general"

def route_query(query: str) -> str:
    qtype = classify_query(query)
    if qtype == "math":
        return "Calculator"
    if qtype in {"knowledge", "general"}:
        return "Retriever"
    if qtype == "creative":
        return "CreativeResponder"
    return "Retriever"

## 6) Grounded retrieval answer

In [ ]:
def split_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", normalize_whitespace(text))
    return [p.strip() for p in parts if p.strip()]

def answer_with_retrieval(query: str, index: SimpleVectorIndex, top_k: int = 3, max_sentences: int = 4) -> Dict[str, Any]:
    results = index.search(query, top_k=top_k)

    sentence_rows = []
    for r in results:
        chunk = r["chunk"]
        sents = split_sentences(chunk.text)
        if not sents:
            continue

        vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
        mat = vec.fit_transform(sents + [query])
        sims = cosine_similarity(mat[-1], mat[:-1]).flatten()

        for sent, sim in zip(sents, sims):
            sentence_rows.append({
                "sentence": sent,
                "local_score": float(sim),
                "chunk_score": float(r["score"]),
                "source": chunk.source,
                "chunk_id": chunk.chunk_id
            })

    sentence_rows = sorted(
        sentence_rows,
        key=lambda x: (x["chunk_score"], x["local_score"]),
        reverse=True
    )

    picked = []
    seen = set()
    for row in sentence_rows:
        key = row["sentence"].lower()
        if key not in seen and len(row["sentence"]) > 35:
            picked.append(row)
            seen.add(key)
        if len(picked) >= max_sentences:
            break

    answer = " ".join(x["sentence"] for x in picked) if picked else "No grounded answer found."

    return {
        "answer": answer,
        "sources": [
            {
                "chunk_id": r["chunk"].chunk_id,
                "source": r["chunk"].source,
                "score": round(r["score"], 4),
                "text": r["chunk"].text
            }
            for r in results
        ]
    }

## 7) Build a miniature agent

In [ ]:
class MiniAgent:
    def __init__(self, tools: Dict[str, Tool], index: SimpleVectorIndex):
        self.tools = tools
        self.index = index

    def run(self, query: str) -> Dict[str, Any]:
        route = route_query(query)

        if route == "Calculator":
            result = self.tools["Calculator"].func(query)
            return {
                "route": route,
                "answer": result,
                "sources": []
            }

        if route == "CreativeResponder":
            return {
                "route": route,
                "answer": f"Creative response placeholder for: {query}",
                "sources": []
            }

        retrieval_output = answer_with_retrieval(query, self.index, top_k=3, max_sentences=4)
        return {
            "route": "Retriever",
            "answer": retrieval_output["answer"],
            "sources": retrieval_output["sources"]
        }

agent = MiniAgent(tools, index)

## 8) Try the agent

In [ ]:
queries = [
    "What is 12 * 12?",
    "What is LangGraph good for?",
    "How do vector databases help RAG?",
    "Which cloud platform is good for Microsoft-centric organizations?"
]

for q in queries:
    out = agent.run(q)
    print("=" * 100)
    print("Query:", q)
    print("Route:", out["route"])
    print("Answer:", out["answer"])
    if out["sources"]:
        print("Sources:", [f"{s['chunk_id']}:{s['source']}" for s in out["sources"]])
    print()

## 9) Inspect provenance

In [ ]:
q = "How do vector databases help RAG?"
out = agent.run(q)

print("ANSWER")
print(out["answer"])
print("\nRETRIEVED SOURCES")
for s in out["sources"]:
    print("=" * 100)
    print(f"{s['chunk_id']} | {s['source']} | score={s['score']}")
    print(s["text"])
    print()

## 10) Observe chunking trade-offs

In [ ]:
def evaluate_configs(query: str, configs: List[Tuple[int, int]]) -> pd.DataFrame:
    rows = []
    for chunk_size, chunk_overlap in configs:
        cfg_chunks = build_chunks(knowledge_base, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        cfg_index = SimpleVectorIndex(cfg_chunks)
        hits = cfg_index.search(query, top_k=3)

        rows.append({
            "chunk_size": chunk_size,
            "chunk_overlap": chunk_overlap,
            "num_chunks": len(cfg_chunks),
            "top_source": hits[0]["chunk"].source if hits else None,
            "top_score": round(hits[0]["score"], 4) if hits else None
        })
    return pd.DataFrame(rows)

evaluate_configs(
    "Why is chunking important in retrieval systems?",
    [(180, 30), (260, 50), (320, 60), (500, 80)]
)

## 11) Simulate the chapter's hybrid routing idea

The chapter describes routing different types of queries to different models or strategies.
Here we simulate that idea with handler labels rather than real external models.

In [ ]:
def hybrid_route_demo(query: str) -> Dict[str, str]:
    qtype = classify_query(query)

    if qtype == "math":
        chosen = "FastMathHandler"
    elif qtype == "knowledge":
        chosen = "GroundedRetrieverHandler"
    elif qtype == "creative":
        chosen = "CreativeStyleHandler"
    else:
        chosen = "GeneralAssistantHandler"

    return {
        "query": query,
        "classified_as": qtype,
        "routed_to": chosen
    }

demo_queries = [
    "Calculate 25 * 7",
    "Explain LangChain tools",
    "Write a short creative intro about AI agents",
    "What are cloud-native agent platforms?"
]

pd.DataFrame([hybrid_route_demo(q) for q in demo_queries])

## 12) What this notebook teaches

This notebook compresses the chapter into four practical engineering lessons:

### A. Tools
Agents become useful when they can call external functions.

### B. Retrieval
Agents become grounded when they can retrieve context from a knowledge base.

### C. Routing
Agents become efficient when they send different query types to the right strategy.

### D. Provenance
Agents become trustworthy when they show which chunks informed the answer.

## 13) How to extend this notebook

To evolve this into a stronger engineering artifact, you can replace components step by step:

1. Replace TF-IDF with embedding models
2. Replace in-memory retrieval with FAISS / Chroma / Pinecone / Weaviate / Qdrant
3. Replace mock web search with a real search API
4. Add metadata filters and reranking
5. Add a real LLM for final grounded synthesis
6. Add a stateful graph workflow similar to LangGraph
7. Add evaluation metrics for retrieval quality and answer quality

## Final takeaway

Chapter 2 is fundamentally about the **toolkit** behind agent engineering.

A practical agent system usually needs all of the following:
- orchestration
- tools
- retrieval
- routing
- observability

This notebook gives you a self-contained working version of those ideas in one place.